# Detección de mislabels por clustering de keypoints

Usa las poses pre-computadas en `pose_per_individual_videos/` para encontrar videos
que probablemente estén mal etiquetados dentro de sus glosas.

## Pipeline
1. **Features**: media + desvío estándar de los 55 keypoints normalizados por video (vector de 220 dims)
2. **Outlier detection within-gloss**: z-score de distancia al centroide de la glosa → identifica mislabels
3. **Visualización global**: PCA + scatter coloreado por score de outlier
4. **Export**: CSV de videos sospechosos rankeados por score

**Primera corrida**: ~10-15 min (21K videos). Las siguientes son instantáneas (cache).

In [ ]:
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import csv
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import pairwise_distances

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
BASE        = Path("..").resolve()
POSE_DIR    = BASE / "data" / "pose_per_individual_videos"
WLASL_JSON  = BASE / "start_kit" / "WLASL_v0.3.json"
CACHE_NPY   = BASE / "data" / "features_cache.npy"
CACHE_META  = BASE / "data" / "features_meta.json"
OUTPUT_CSV  = BASE / "data" / "mislabels_sospechosos.csv"

# Upper body joint indices en BODY_25 (13 joints, igual que el paper)
UPPER_BODY_IDX = [0, 1, 2, 3, 4, 5, 6, 7, 8, 15, 16, 17, 18]

N_WORKERS   = 8   # threads para lectura paralela de JSONs
OUTLIER_Z   = 2.5 # umbral de z-score para marcar un video como sospechoso

## 1 — Metadata: mapeo video_id → glosa

In [ ]:
with open(WLASL_JSON) as f:
    wlasl = json.load(f)

# video_id → {gloss, split, signer_id, source}
video_meta = {}
for entry in wlasl:
    for inst in entry["instances"]:
        video_meta[inst["video_id"]] = {
            "gloss":     entry["gloss"],
            "split":     inst["split"],
            "signer_id": inst["signer_id"],
            "source":    inst["source"],
        }

# Sólo los que tienen carpeta de poses
available_ids = {d.name for d in POSE_DIR.iterdir() if d.is_dir()}
valid_ids     = [vid for vid in video_meta if vid in available_ids]

print(f"Videos en WLASL JSON:       {len(video_meta)}")
print(f"Con poses disponibles:      {len(valid_ids)}")
print(f"Glosas únicas:              {len({video_meta[v]['gloss'] for v in valid_ids})}")

## 2 — Extracción de features (con cache)

Por cada video:
- Carga todos los frames (JSONs de poses)
- Normaliza keypoints: centra en cuello (joint 1), escala por distancia cuello→cadera (joint 8)
- Extrae 55 joints × 2 coords = 110 valores por frame
- Agrega temporalmente: media + desvío = vector de **220 dims** por video

Frames donde el cuello no está detectado se ignoran.

In [ ]:
def load_frame_keypoints(json_path: Path):
    """Carga un JSON de keypoints y devuelve (pose_25x2, hand_l_21x2, hand_r_21x2) o None."""
    with open(json_path) as f:
        data = json.load(f)
    if not data["people"]:
        return None
    p = data["people"][0]
    pose   = np.array(p["pose_keypoints_2d"]).reshape(25, 3)[:, :2]    # (25, 2)
    hand_l = np.array(p["hand_left_keypoints_2d"]).reshape(21, 3)[:, :2]  # (21, 2)
    hand_r = np.array(p["hand_right_keypoints_2d"]).reshape(21, 3)[:, :2] # (21, 2)
    return pose, hand_l, hand_r


def extract_video_feature(video_id: str) -> np.ndarray | None:
    """
    Devuelve un vector de 220 dimensiones:
    media y desvío de los 55 keypoints normalizados a lo largo del video.
    Devuelve None si el video no tiene frames válidos.
    """
    video_dir   = POSE_DIR / video_id
    frame_files = sorted(video_dir.glob("image_*_keypoints.json"))
    if not frame_files:
        return None

    frames = []
    for fp in frame_files:
        kp = load_frame_keypoints(fp)
        if kp is None:
            continue
        pose, hand_l, hand_r = kp

        neck   = pose[1]   # joint 1 = Neck
        midhip = pose[8]   # joint 8 = MidHip

        # Saltear frames sin detección
        if np.all(neck == 0) or np.all(midhip == 0):
            continue
        scale = np.linalg.norm(neck - midhip)
        if scale < 1e-6:
            continue

        # Normalizar: centrar en cuello, escalar por altura del torso
        upper  = (pose[UPPER_BODY_IDX] - neck) / scale    # (13, 2)
        hand_l = (hand_l - neck) / scale                   # (21, 2)
        hand_r = (hand_r - neck) / scale                   # (21, 2)

        frame_vec = np.concatenate([upper.ravel(), hand_l.ravel(), hand_r.ravel()])  # (110,)
        frames.append(frame_vec)

    if not frames:
        return None

    arr = np.stack(frames)  # (N_frames, 110)
    return np.concatenate([arr.mean(0), arr.std(0)])  # (220,)


def progress_bar(current, total, start_time, width=40):
    pct  = current / total
    done = int(width * pct)
    bar  = '=' * done + '-' * (width - done)
    elapsed = time.time() - start_time
    eta = (elapsed / max(current, 1)) * (total - current)
    print(f"\r[{bar}] {current}/{total} ({pct*100:.1f}%) | ETA: {eta:.0f}s  ",
          end='', flush=True)

In [ ]:
if CACHE_NPY.exists() and CACHE_META.exists():
    print("Cargando features desde cache...")
    features_arr = np.load(CACHE_NPY)
    with open(CACHE_META) as f:
        cache_meta = json.load(f)
    cached_ids = cache_meta["video_ids"]
    print(f"Cache cargado: {len(cached_ids)} videos, features shape: {features_arr.shape}")

else:
    print(f"Extrayendo features de {len(valid_ids)} videos ({N_WORKERS} workers)...")
    print("Esto tarda ~10-15 min la primera vez.\n")

    results = {}  # video_id → feature vector
    failed  = []
    done    = 0
    t0      = time.time()

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(extract_video_feature, vid): vid for vid in valid_ids}
        for future in as_completed(futures):
            vid  = futures[future]
            feat = future.result()
            done += 1
            if feat is not None:
                results[vid] = feat
            else:
                failed.append(vid)
            if done % 100 == 0 or done == len(valid_ids):
                progress_bar(done, len(valid_ids), t0)

    print(f"\n\nCompletado en {(time.time()-t0)/60:.1f} min")
    print(f"Videos con features válidas: {len(results)} / {len(valid_ids)}")
    print(f"Videos sin detección (descartados): {len(failed)}")

    # Guardar cache
    cached_ids   = list(results.keys())
    features_arr = np.stack([results[v] for v in cached_ids])  # (N, 220)
    np.save(CACHE_NPY, features_arr)
    with open(CACHE_META, "w") as f:
        json.dump({"video_ids": cached_ids}, f)
    print(f"Cache guardado: {CACHE_NPY}")

## 3 — Detección de mislabels (within-gloss outlier score)

Para cada glosa:
1. Calcula el centroide (media de features) de todos sus videos
2. Calcula la distancia de cada video al centroide
3. Z-normaliza esas distancias dentro de la glosa

Un video con z-score > 2.5 está a más de 2.5 desvíos del centro de su glosa → sospechoso.

In [ ]:
# Construir arrays indexados por posición
ids     = np.array(cached_ids)
glosses = np.array([video_meta[v]["gloss"] for v in cached_ids])
splits  = np.array([video_meta[v]["split"]  for v in cached_ids])

unique_glosses = np.unique(glosses)
print(f"Features cargadas: {len(ids)} videos, {len(unique_glosses)} glosas")

# ── Within-gloss outlier score ─────────────────────────────────────────────
outlier_scores = np.zeros(len(ids))   # z-score de distancia al centroide
dist_to_center = np.zeros(len(ids))   # distancia euclidiana bruta
gloss_size     = np.zeros(len(ids), dtype=int)

for gloss in unique_glosses:
    mask  = glosses == gloss
    idx   = np.where(mask)[0]
    X     = features_arr[idx]          # (k, 220)
    k     = len(idx)
    gloss_size[idx] = k

    if k == 1:
        outlier_scores[idx] = 0.0
        continue

    centroid = X.mean(axis=0)           # (220,)
    dists    = np.linalg.norm(X - centroid, axis=1)  # (k,)
    dist_to_center[idx] = dists

    # Z-score dentro de la glosa
    mu  = dists.mean()
    std = dists.std()
    if std < 1e-9:
        outlier_scores[idx] = 0.0
    else:
        outlier_scores[idx] = (dists - mu) / std

is_outlier = outlier_scores > OUTLIER_Z
print(f"\nVideos marcados como sospechosos (z > {OUTLIER_Z}): {is_outlier.sum()}")
print(f"Porcentaje del dataset: {is_outlier.mean()*100:.1f}%")

# Glosas con más outliers
from collections import Counter
outlier_by_gloss = Counter(glosses[is_outlier])
print("\nTop 10 glosas con más videos sospechosos:")
for gloss, cnt in outlier_by_gloss.most_common(10):
    total_in_gloss = (glosses == gloss).sum()
    print(f"  {gloss:20s}: {cnt}/{total_in_gloss} videos sospechosos")

## 4 — Embedding global con PCA

Reduce las 220 dims a 2D para visualizar la distribución de todos los videos.
Para una vista más fina, correr t-SNE en la celda siguiente (más lento).

In [ ]:
print("Calculando PCA...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_arr)  # (N, 220)

pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)            # (N, 50)
print(f"Varianza explicada (50 comp): {pca.explained_variance_ratio_.sum()*100:.1f}%")

# PCA a 2D para visualización directa
pca2 = PCA(n_components=2, random_state=42)
X_2d_pca = pca2.fit_transform(X_scaled)        # (N, 2)
print(f"Varianza explicada (2 comp):  {pca2.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# ── Visualización global: PCA 2D coloreada por outlier score ───────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel izquierdo: todos los videos, coloreados por z-score
ax = axes[0]
normal = ~is_outlier
sc = ax.scatter(
    X_2d_pca[normal, 0], X_2d_pca[normal, 1],
    c=outlier_scores[normal], cmap='Blues',
    s=2, alpha=0.3, label='Normal'
)
ax.scatter(
    X_2d_pca[is_outlier, 0], X_2d_pca[is_outlier, 1],
    c='red', s=8, alpha=0.6, label=f'Sospechoso (z>{OUTLIER_Z})', zorder=5
)
ax.set_title('PCA 2D — coloreado por outlier score', fontsize=11)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.colorbar(sc, ax=ax, label='z-score')
ax.legend(fontsize=8)

# Panel derecho: distribución de outlier scores
ax2 = axes[1]
ax2.hist(outlier_scores, bins=100, color='steelblue', alpha=0.7, label='Todos')
ax2.axvline(OUTLIER_Z, color='red', linestyle='--', label=f'Umbral z={OUTLIER_Z}')
ax2.set_xlabel('Z-score de distancia al centroide de la glosa')
ax2.set_ylabel('# videos')
ax2.set_title('Distribución de outlier scores')
ax2.legend()

plt.tight_layout()
plt.savefig(BASE / 'data' / 'clustering_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — t-SNE (opcional, más lento pero mejor separación)

Correr esta celda sólo si querés la visualización de mayor calidad. Tarda ~5-10 min con 21K videos.

In [ ]:
TSNE_CACHE = BASE / 'data' / 'tsne_embedding.npy'

if TSNE_CACHE.exists():
    print("Cargando t-SNE desde cache...")
    X_2d_tsne = np.load(TSNE_CACHE)
else:
    print(f"Calculando t-SNE sobre {len(ids)} videos (puede tardar ~5-10 min)...")
    t0 = time.time()
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        n_iter=1000,
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    X_2d_tsne = tsne.fit_transform(X_pca)  # usa las 50 componentes de PCA
    np.save(TSNE_CACHE, X_2d_tsne)
    print(f"t-SNE completado en {(time.time()-t0)/60:.1f} min")

# Mismo scatter coloreado por outlier score
fig, ax = plt.subplots(figsize=(12, 10))
ax.scatter(
    X_2d_tsne[normal, 0], X_2d_tsne[normal, 1],
    c='steelblue', s=1, alpha=0.2, label='Normal'
)
ax.scatter(
    X_2d_tsne[is_outlier, 0], X_2d_tsne[is_outlier, 1],
    c='red', s=6, alpha=0.7, label=f'Sospechoso (z>{OUTLIER_Z})', zorder=5
)
ax.set_title('t-SNE — videos sospechosos en rojo', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(BASE / 'data' / 'tsne_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Inspección por glosa

Para una glosa en particular: muestra sus videos en el espacio PCA y destaca los sospechosos.

In [ ]:
def inspect_gloss(gloss_name: str, embedding=X_2d_pca, embedding_name='PCA'):
    """Visualiza los videos de una glosa en el embedding global y destaca outliers."""
    mask         = glosses == gloss_name
    mask_out     = mask & is_outlier
    mask_ok      = mask & ~is_outlier

    if not mask.any():
        print(f"Glosa '{gloss_name}' no encontrada.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel izq: posición de la glosa en el espacio global
    ax = axes[0]
    ax.scatter(embedding[:, 0], embedding[:, 1], c='lightgray', s=1, alpha=0.2)
    ax.scatter(embedding[mask_ok, 0], embedding[mask_ok, 1],
               c='steelblue', s=30, zorder=4, label='Normal')
    ax.scatter(embedding[mask_out, 0], embedding[mask_out, 1],
               c='red', s=60, marker='x', zorder=5, linewidths=1.5,
               label=f'Sospechoso (z>{OUTLIER_Z})')
    ax.set_title(f"{embedding_name} — glosa: '{gloss_name}'")
    ax.legend(fontsize=8)

    # Panel der: outlier score por video
    ax2 = axes[1]
    vids_in_gloss = ids[mask]
    scores        = outlier_scores[mask]
    split_labels  = splits[mask]
    colors        = {'train': 'steelblue', 'val': 'orange', 'test': 'green'}
    bar_colors    = [('red' if s > OUTLIER_Z else colors.get(sp, 'gray'))
                     for s, sp in zip(scores, split_labels)]
    ax2.barh(range(len(vids_in_gloss)), scores, color=bar_colors)
    ax2.axvline(OUTLIER_Z, color='red', linestyle='--', alpha=0.7)
    ax2.set_yticks(range(len(vids_in_gloss)))
    ax2.set_yticklabels(vids_in_gloss, fontsize=7)
    ax2.set_xlabel('Outlier z-score')
    ax2.set_title('Score por video (rojo = sospechoso, azul/naranja/verde = split)')

    plt.tight_layout()
    plt.show()

    # Tabla de sospechosos
    out_mask  = scores > OUTLIER_Z
    if out_mask.any():
        print(f"\nVideos sospechosos en '{gloss_name}':")
        for vid, score, sp in sorted(
            zip(vids_in_gloss[out_mask], scores[out_mask], split_labels[out_mask]),
            key=lambda x: -x[1]
        ):
            print(f"  video_id={vid}  split={sp}  z={score:.2f}")
    else:
        print(f"  Ningún sospechoso en '{gloss_name}'.")


# Ejemplo: inspeccionar las glosas con más sospechosos
top_gloss = outlier_by_gloss.most_common(1)[0][0]
inspect_gloss(top_gloss)

In [ ]:
# ── Celda interactiva: cambiar la glosa a inspeccionar ─────────────────────
# Editar esta línea y re-ejecutar la celda:
inspect_gloss("computer")

## 7 — Export CSV de videos sospechosos

Tabla rankeada de mayor a menor z-score, con todos los datos necesarios para revisar manualmente.

In [ ]:
# Ordenar por z-score descendente
order = np.argsort(-outlier_scores)

rows = []
for i in order:
    if outlier_scores[i] <= OUTLIER_Z:
        break  # el resto son normales
    meta = video_meta.get(ids[i], {})
    rows.append({
        "video_id":     ids[i],
        "gloss":        glosses[i],
        "outlier_z":    round(float(outlier_scores[i]), 3),
        "dist_centroid":round(float(dist_to_center[i]), 3),
        "split":        splits[i],
        "signer_id":    meta.get("signer_id", ""),
        "source":       meta.get("source", ""),
        "gloss_size":   int(gloss_size[i]),
    })

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"CSV guardado: {OUTPUT_CSV}")
print(f"Total de videos sospechosos: {len(rows)}")
print(f"\nTop 20 más sospechosos:")
print(f"{'video_id':>10}  {'gloss':25}  {'z-score':>8}  {'split':>8}  {'fuente'}")
print('-' * 75)
for r in rows[:20]:
    print(f"{r['video_id']:>10}  {r['gloss']:25}  {r['outlier_z']:>8.3f}  {r['split']:>8}  {r['source']}")

## 8 — Análisis de glosas problemáticas

¿Qué glosas tienen consistentemente alta varianza interna? Eso puede indicar:
- Múltiples variantes dialectales mezcladas
- Videos de baja calidad (sin detección)
- Mislabels sistemáticos

In [ ]:
# Varianza intra-glosa (media de distancias al centroide)
gloss_stats = []
for gloss in unique_glosses:
    mask  = glosses == gloss
    n     = mask.sum()
    if n < 3:
        continue
    dists      = dist_to_center[mask]
    n_outliers = is_outlier[mask].sum()
    gloss_stats.append({
        "gloss":       gloss,
        "n_videos":    int(n),
        "n_outliers":  int(n_outliers),
        "pct_outlier": float(n_outliers / n),
        "mean_dist":   float(dists.mean()),
        "max_dist":    float(dists.max()),
    })

gloss_stats.sort(key=lambda x: -x["pct_outlier"])

print("Top 20 glosas más problemáticas (% de videos sospechosos):")
print(f"{'glosa':25}  {'n':>5}  {'outliers':>8}  {'%':>6}  {'dist_media':>10}")
print('-' * 65)
for g in gloss_stats[:20]:
    print(f"{g['gloss']:25}  {g['n_videos']:>5}  {g['n_outliers']:>8}  "
          f"{g['pct_outlier']*100:>6.1f}%  {g['mean_dist']:>10.3f}")

# Gráfico: distribución de % outliers por glosa
pcts = [g['pct_outlier'] for g in gloss_stats]
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pcts, bins=50, color='steelblue', alpha=0.8)
ax.set_xlabel('Fracción de videos sospechosos por glosa')
ax.set_ylabel('# glosas')
ax.set_title('Distribución de la "tasa de contaminación" por glosa')
plt.tight_layout()
plt.savefig(BASE / 'data' / 'gloss_contamination.png', dpi=150, bbox_inches='tight')
plt.show()